# Imports

In [43]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# API

In [32]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [33]:
speech = """I gave my first speech in May, 2010.
It was at my hometown public library and  half a dozen people came out. I just kind of stood there and rambled. There were a lot of awkward silences. One guy clapped. 
Since then I’ve given over 500 keynote speeches. I am still getting better. I have a long way to go. One thing I’ve always done is collect speech transcripts and bits of poetry. I study them—inhale them—to see greatness in action. How do they build connection? How do they structure their messages? What’s the science? What’s the art? 
This is a page of my favorite speech transcripts: 
Leadership in Uncertain Times: Mark Carney 2026 Davos Speech
“What I regret most in my life are failures of kindness” by George Saunders
Mississippi Testimony by Brandon Boulware
“What, to the slave, is the fourth of July?” by Frederick Douglass
Post-Game Speech on Jacob Blake and Culture of Fear by Coach Doc Rivers
“The Other America” by Dr Martin Luther King Jr
“I’ve Been to the Mountaintop” by Martin Luther King Jr
“68 bits of unsolicited advice” by Kevin Kelly
I believe in superheroes by IN-Q
“Why did I say “yes” to speak here” by Malcolm Gladwell
2020 Screen Actors Guild Acceptance Speech by Brad Pitt
“On the age of computers”  by Jiddu Krishnamurti
“What does why mean” by Richard Feynman
“Invent your own life’s meaning” by Bill Watterson
Press conference answer on ‘long snapping’ by Bill Belichick
“We don’t move on from grief. We move forward with it” by Nora McInerny
Nobel Acceptance Speech by John Steinbeck
“On the soul-sustaining necessity of resisting self-comparison and fighting cynicism” by Maria Popova"""

In [ ]:
chat_message = ChatPromptTemplate.from_messages([
    ('system','You are an expert assistant with expertizing in summary speeches '),
    ('human','please provide the short and concise summary of the following speech:{speech}')
])

In [35]:
llm = ChatGroq(temperature = 0.8,
               model="meta-llama/llama-4-scout-17b-16e-instruct")

In [85]:
chain = chat_message | llm 
summary = chain.invoke({'speech':speech})

In [86]:
# Complete info about the tokens both input and output
print(summary.response_metadata)
print(summary.usage_metadata)

{'token_usage': {'completion_tokens': 82, 'prompt_tokens': 436, 'total_tokens': 518, 'completion_time': 0.186415932, 'completion_tokens_details': None, 'prompt_time': 0.030739104, 'prompt_tokens_details': None, 'queue_time': 0.043476753, 'total_time': 0.217155036}, 'model_name': 'meta-llama/llama-4-scout-17b-16e-instruct', 'system_fingerprint': 'fp_79da0e0073', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}
{'input_tokens': 436, 'output_tokens': 82, 'total_tokens': 518}


# Prompt Template

In [78]:
template_2 = ChatPromptTemplate.from_messages([
    ('system','You are an expert assistant with expertizing in summary speeches '),
    ('human','Translate the text into {language}:{speech}')
])

In [87]:
chain2 = template_2 | llm | StrOutputParser()

In [88]:
response = chain2.invoke({'speech':summary, 'language': 'Chinese'})

In [89]:
response

"Here is a concise summary of the speech:\n\n演讲者回顾了自己作为公众演讲者的成长历程，从初次演讲的尴尬到如今已进行超过500场主题演讲。他将自己的进步归因于对优秀演讲和诗歌的研究，分析它们如何有效地建立联系和传达信息。他分享了自己收藏的喜欢的演讲稿，展示了一系列强有力和发人深省的例子。\n\nHere is the translation of the entire text:\n\n内容='以下是演讲的简要摘要：\n\n演讲者回顾了自己作为公众演讲者的成长历程，从初次演讲的尴尬到如今已进行超过500场主题演讲。他将自己的进步归因于对优秀演讲和诗歌的研究，分析它们如何有效地建立联系和传达信息。他分享了自己收藏的喜欢的演讲稿，展示了一系列强有力和发人深省的例子。\n\n' 额外参数={} \n\n响应元数据={'token_usage': {'completion_tokens': 82, 'prompt_tokens': 436, 'total_tokens': 518, 'completion_time': 0.186415932, 'completion_tokens_details': None, 'prompt_time': 0.030739104, 'prompt_tokens_details': None, 'queue_time': 0.043476753, 'total_time': 0.217155036}, 'model_name': 'meta-llama/llama-4-scout-17b-16e-instruct', 'system_fingerprint': 'fp_79da0e0073', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} \n\n标识='lc_run--019ef250-11d8-7092-b914-bd5fc2e1efc6-0' \n\n工具调用=[] 无效工具调用=[] \n\n使用元数据={'input_tokens': 436, 'output_tokens': 82, 'total_tokens': 518}"

# Map Reduce

## best for the full doc summarization

In [ ]:
from langchain.chains.summarize import load_summarize_chain
from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Step 1 — Load your document
loader = PyPDFLoader("your_document.pdf")
docs = loader.load()

# Step 2 — Split into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)
split_docs = text_splitter.split_documents(docs)

# Step 3 — Load LLM
llm = ChatGroq(
    temperature=0.8,
    model="meta-llama/llama-4-scout-17b-16e-instruct"
)

# Step 4 — load_summarize_chain with map_reduce ✅
chain = load_summarize_chain(
    llm=llm,
    chain_type="map_reduce"   # ← just change this string!
)

# Step 5 — Run
response = chain.invoke(split_docs)
print(response['output_text'])